In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
Number of GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [5]:
script = '''
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler


# ── Model ────────────────────────────────────────────────────────────────────

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size=1000, hidden=256, num_layers=4, num_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=num_heads,
            dim_feedforward=hidden * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = self.transformer(x)
        return self.head(x)


# ── Dataset ───────────────────────────────────────────────────────────────────

def make_dataset(num_samples=1024, seq_len=64, vocab_size=1000):
    x = torch.randint(0, vocab_size, (num_samples, seq_len))
    y = torch.randint(0, vocab_size, (num_samples, seq_len))
    return TensorDataset(x, y)


# ── Training ──────────────────────────────────────────────────────────────────

def train():
    # Init — reads RANK, WORLD_SIZE, MASTER_ADDR, MASTER_PORT from torchrun
    dist.init_process_group(backend="nccl")
    rank       = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")

    # Only rank 0 prints — otherwise output gets duplicated
    def log(msg):
        if rank == 0:
            print(msg, flush=True)

    log(f"Training on {world_size} GPUs")
    log(f"  GPU 0: {torch.cuda.get_device_name(0)}")
    log(f"  GPU 1: {torch.cuda.get_device_name(1)}")
    log("")

    # Model — DDP broadcasts rank 0 weights to all GPUs at construction
    model = TinyTransformer().to(device)
    model = DDP(model, device_ids=[rank])
    log(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Data — DistributedSampler splits dataset across GPUs, no overlap
    dataset = make_dataset()
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader  = DataLoader(dataset, batch_size=32, sampler=sampler, pin_memory=True)
    log(f"Batches per GPU per epoch: {len(loader)}")
    log("")

    # Optimizer and loss
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()

    # Training loop
    epochs = 5
    for epoch in range(epochs):
        model.train()
        sampler.set_epoch(epoch)   # reshuffle differently each epoch

        total_loss  = torch.tensor(0.0, device=device)
        num_batches = 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)                                  # [B, seq, vocab]
            loss   = criterion(logits.view(-1, 1000), y.view(-1))
            loss.backward()                                    # AllReduce happens here
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss  += loss.detach()
            num_batches += 1

        # Average loss across both GPUs for accurate logging
        dist.reduce(total_loss, dst=0, op=dist.ReduceOp.AVG)
        log(f"Epoch {epoch+1}/{epochs}  |  loss: {total_loss.item()/num_batches:.4f}")

    # Save checkpoint — model.module strips the DDP wrapper
    if rank == 0:
        torch.save(model.module.state_dict(), "/kaggle/working/checkpoint.pt")
        log("\\nCheckpoint saved to /kaggle/working/checkpoint.pt")
    peak_mem = torch.cuda.max_memory_allocated(rank) / 1e9
    print(f"[Rank {rank}] GPU {rank} peak memory: {peak_mem:.3f} GB", flush=True)

    # Barrier ensures all ranks print before any process exits
    dist.barrier()

    dist.destroy_process_group()


if __name__ == "__main__":
    train()
'''

# Write script to disk
with open("/kaggle/working/train_ddp.py", "w") as f:
    f.write(script)

print("Script written to /kaggle/working/train_ddp.py")

Script written to /kaggle/working/train_ddp.py


In [6]:
!torchrun --nproc_per_node=2 /kaggle/working/train_ddp.py

W0629 21:56:38.632000 151 torch/distributed/run.py:852] 
W0629 21:56:38.632000 151 torch/distributed/run.py:852] *****************************************
W0629 21:56:38.632000 151 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0629 21:56:38.632000 151 torch/distributed/run.py:852] *****************************************
Training on 2 GPUs
  GPU 0: Tesla T4
  GPU 1: Tesla T4

Model parameters: 3,672,040
Batches per GPU per epoch: 16

Epoch 1/5  |  loss: 7.0011
Epoch 2/5  |  loss: 6.9227
Epoch 3/5  |  loss: 6.9196
Epoch 4/5  |  loss: 6.9149
Epoch 5/5  |  loss: 6.9113
[Rank 1] GPU 1 peak memory: 0.271 GB

Checkpoint saved to /kaggle/working/checkpoint.pt
[Rank 0] GPU 0 peak memory: 0.271 GB
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using

In [4]:
# Check checkpoint was saved
import os
print(f"Checkpoint exists: {os.path.exists('/kaggle/working/checkpoint.pt')}")

# Check GPU memory was used on both
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.max_memory_allocated(i) / 1e9
    print(f"GPU {i} peak memory used: {mem:.3f} GB")

Checkpoint exists: True
GPU 0 peak memory used: 0.000 GB
GPU 1 peak memory used: 0.000 GB
